In [0]:
%run ../../config/metadata_setup

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, DecimalType, TimestampType
from pyspark.sql.functions import col, lit, filter, to_date, current_timestamp
import os

In [0]:
class GoldSalesFactsStreaming:
    def __init__(self, spark):
        self.spark = spark
        self.catalog_name = CONFIG["catalog"]["name"]
        self.schema_name = CONFIG["catalog"]["schema"]

        # Paths
        self.schema_path = CONFIG["paths"]["schemas"]
        self.checkpoint_path = CONFIG["paths"]["checkpoints"]

        # up-stream table name
        self.upstream_order_details_table = TABLES["silver"]["order_details"]
        self.upstream_orders_table = TABLES["silver"]["orders"]

        # down-stream table name
        self.downstream_table_name = TABLES["gold"]["sales_facts"]

    # --------------------------------
    # Read silver order details streaming data
    # --------------------------------
    def read_order_details_stream(self):
        return self.spark.readStream.table(
            f"{self.catalog_name}.{self.schema_name}.{self.upstream_order_details_table}"
        )

    # --------------------------------
    # Read silver orders streaming data
    # --------------------------------
    def read_orders_stream(self):
        return self.spark.readStream.table(
            f"{self.catalog_name}.{self.schema_name}.{self.upstream_orders_table}"
        )

    # -------------------------------
    # upsert join data in gold table
    # -------------------------------
    def upsert_to_gold(self, microBatchDF, batchId):
        microBatchDF.createOrReplaceTempView("v_sales_facts")

        microBatchDF.sparkSession.sql(
            f"""
              MERGE INTO {self.catalog_name}.{self.schema_name}.{self.downstream_table_name} AS t
              USING v_sales_facts AS s
              ON t.order_id = s.order_id
              WHEN MATCHED THEN                        
        """
        )

    # ------------------------------------------------
    # steaming order join with streaming order details
    # ------------------------------------------------
    def process_stream(self):
        order_df = self.read_orders_stream()
        order_details_df = self.read_order_details_stream()
        return (
            order_df.alias("order")
            .join(
                order_details_df.alias("order_details"),
                col("order.order_id") == col("order_details.order_id"),
            )
            .drop("_rescued_data")
            .withColumn("amount", col("order_details.amount").cast(DecimalType(18, 2)))
            .withColumn("profit", col("order_details.profit").cast(DecimalType(18, 2)))
            .withColumn("quantity", col("order_details.quantity").cast("int"))
            .withColumn("insert_timestamp", current_timestamp())
            .withColumn("update_timestamp", lit(None).cast(TimestampType()))
            .select(
                col("order.order_id"),
                col("order.order_date"),
                col("order.customer_name"),
                col("order.state"),
                col("order.city"),
                "amount",
                "profit",
                "quantity",
                col("order_details.category"),
                col("order_details.sub_category"),
                "insert_timestamp",
                "update_timestamp"
            )
        )

    # --------------------------------
    # Only for droping table
    #---------------------------------
    #def drop_table(self):
    #    dbutils.fs.rm(f"{self.checkpoint_path}/_{self.downstream_table_name}", True)

    # ---------------------------------
    # write to delta table (streaming)
    # ---------------------------------
    def write_stream(self, df):
        try:
            query = (
                df.writeStream.option(
                    "checkpointLocation",
                    f"{self.checkpoint_path}/_{self.downstream_table_name}",
                )
                # .foreachBatch(self.upsert_to_gold)
                .trigger(availableNow=True)
                .option("mergeSchema", "true")
                .table(
                    f"{self.catalog_name}.{self.schema_name}.{self.downstream_table_name}"
                )
            )
            query.awaitTermination()
        except Exception as e:
            print(e)

    def run(self):
        df = self.process_stream()
        self.write_stream(df)

In [0]:
obj = GoldSalesFactsStreaming(spark)
obj.run()